# 1AMMV5 Detailed Walkthrough

This notebook simulates **AI-managed liquidity provision** in a constant-product AMM and compares it against a static LP baseline. The full script is organized into 6 sections and runs end-to-end through `main()`.

## 1) High-level objective

The experiment answers a practical question:

- If an LP can dynamically adjust liquidity placement and exposure with an RL policy, can it improve value/fees and reduce risk versus a passive LP?

The code evaluates this by running many episodes and comparing:

- final LP portfolio value
- average impermanent loss (IL)
- fees earned
- AMM price efficiency vs external market (price dislocation)

## 2) Section-by-section architecture

### Section 0: Imports and setup

- Loads `numpy`, `random`, replay buffer utilities (`deque`), plotting (`matplotlib`), and optional data fetch (`requests`).
- Uses `matplotlib.use("Agg")` so plots save to file in non-interactive runs.

### Section 1: `AMMEnvironment`

This is the core market microstructure model. It tracks pool reserves (`token_a`, `token_b`) and simulates three interacting processes each step:

1. **External market move** (`_next_market_price`)
   - Uses provided `price_feed` if available, otherwise stochastic dynamics.
   - Supports regime changes in volatility/drift every fixed interval.
   - Sentiment slightly tilts drift (does not fully control price).

2. **Organic user flow** (`_simulate_order_flow`)
   - Draws noisy trade notional sizes (lognormal).
   - Uses sentiment-skewed buy/sell probability.
   - Executes swaps with fee collection and tracks traded notional in token-B terms.

3. **Arbitrage flow** (`_run_arbitrage`)
   - If AMM price differs from market price by more than `arbitrage_band_bps`, arbitrage trades partially close the gap.
   - Includes friction floor (`arb_gas_cost_b`) so tiny mispricings are ignored.

Each `simulate_market_activity()` step records:

- AMM price and market price histories
- per-step volume and fees
- arbitrage volume
- dislocation = `(amm_price - market_price)/market_price`

### Section 2: `LiquidityProvider`

Represents an LP position and valuation logic.

- `calculate_impermanent_loss(current_price)`: standard CP-AMM IL formula.
- `calculate_position_value(current_price, active_exposure, trading_costs)`
  - starts from HODL benchmark value
  - applies IL scaled by active exposure
  - adds earned fees
  - subtracts explicit trading/rebalance costs

This makes valuation execution-aware (important for active strategies).

### Section 3: `SentimentAnalyzer`

Generates a sentiment signal in `[-1, 1]` from recent price behavior (or a deterministic sentiment feed).

- Computes momentum/volatility proxy from recent returns.
- Adds noise to mimic imperfect NLP signal.
- Applies EMA smoothing so sentiment is not unrealistically jumpy.

### Section 4: `DQLAgent`

Implements a lightweight Deep Q-Learning style agent using a linear function approximator.

- Action space (5 actions):
  - hold
  - increase liquidity
  - decrease liquidity
  - recenter range up
  - recenter range down

- State vector (8 features):
  - AMM return
  - external market return
  - sentiment
  - liquidity ratio
  - current IL
  - normalized fees earned
  - recent volatility
  - AMM-market dislocation

- Learning mechanics:
  - epsilon-greedy exploration
  - replay memory
  - one-step TD target updates on linear weights

- Reward is **cost-aware** and multi-objective:
  - rewards value growth + fee yield
  - penalizes rising IL
  - penalizes market dislocation
  - penalizes rebalance cost/churn

### Section 5: `AIAMMSimulation`

Coordinates experiments and comparisons.

Helper methods:
- `_pool_share(...)`: LP share approximation from initial deposits vs pool reserves.
- `_recent_volatility(...)`: rolling return volatility.
- `_active_fraction(...)`: approximates concentrated-liquidity behavior (in-range gets high utilization; out-of-range decays smoothly).

Main runs:
- `run_static_strategy()`:
  - LP does not actively adjust controls.
  - earns pool-proportional fees from actual simulated flow.

- `run_ai_strategy()`:
  - agent observes state, chooses action, and updates policy each step.
  - manages both `liquidity_ratio` and `range_center`.
  - pays per-action rebalance cost.
  - receives fees based on active fraction and pool share.
  - computes reward and trains from transition replay.

- `run_comparison()`:
  - runs both strategies across many episodes.
  - aggregates summary metrics and volatility stats.
  - prints action distribution + learning diagnostics.
  - calls `generate_plots(...)` and saves `aiamm_results.png`.

### Section 6: `main()` execution

- Optional switch to load CoinGecko OHLC and use real close prices as market feed.
- By default runs synthetic market path.
- Instantiates simulation and executes the full comparison pipeline.

## 3) Data/control flow per AI timestep

For each AI step, the script does:

1. sentiment update
2. environment step (market move + flow + arbitrage)
3. compute IL, volatility, active fraction
4. build state vector
5. choose action (epsilon-greedy)
6. apply action and rebalance cost
7. compute earned fees and current LP value
8. compute reward
9. store transition and replay-train
10. append metrics/history for analysis and plotting

## 4) Why V5 is significant

Compared to simpler AMM toy setups, V5 introduces market microstructure features that make outcomes more realistic:

- explicit market-vs-AMM dislocation
- arbitrage correction with friction
- fee generation from actual simulated order flow (not only synthetic random fees)
- active-liquidity utilization logic
- cost-aware reward shaping

These changes make the static-vs-AI comparison more meaningful and closer to real LP tradeoffs.

## 5) Practical note

The default `main()` configuration is computationally heavy (`num_steps=5000`, `num_episodes=200`). For quick iteration in notebooks, use smaller values first, then scale up for final metrics.


In [1]:
# %% [markdown]
# # AIAMM Simulation Notebook Script
# This file is structured with Jupyter cell markers for easy execution in a
# notebook environment (e.g., JupyterLab, VS Code).
#
# References for execution order:
# - Section 1: AMM Environment
# - Section 2: Liquidity Provider Simulation
# - Section 3: Sentiment Analysis (Simulated)
# - Section 4: DQL Agent for Liquidity OptimizationÃ³
# - Section 5: Simulation Engine
# - Section 6: Main Execution

# %% [markdown]
# ## Section 0: Imports & Setup (Run First)

# %%
# Core numerics + reproducibility.
import numpy as np
import random
from collections import deque
# Plotting and optional real-market data fetch.
import matplotlib
import requests

matplotlib.use("Agg")  # Use non-interactive backend for notebook execution
import matplotlib.pyplot as plt

# Pipeline overview:
# 1) AMMEnvironment generates AMM + external market dynamics.
# 2) SentimentAnalyzer emits a signal used by market activity.
# 3) LiquidityProvider converts pool dynamics into LP value/IL/fee metrics.
# 4) DQLAgent observes state, chooses actions, and learns from rewards.
# 5) AIAMMSimulation orchestrates episodes, compares static vs AI, and plots.
# Lets the same code run cleanly in notebook mode and script mode.
FILE_LABEL = globals().get("__file__", "notebook")
print("RUNNING FILE", FILE_LABEL)
print("Section 0 done")

# %% [markdown]
# ## Section 1: AMM Environment
# Reference: `AMMEnvironment`

# %%


class AMMEnvironment:
    """
    Simulates a constant-product AMM with:
    - an external reference market price
    - organic order flow
    - arbitrage that pulls AMM price toward market price
    """

    def __init__(
        self,
        initial_token_a=10000,
        initial_token_b=10000,
        fee=0.003,
        price_feed=None,
        arbitrage_band_bps=12,
        max_trade_fraction=0.12,
        arb_gas_cost_b=0.25,
        rebalance_gas_cost_b=0.15,
    ):
        # Initial pool reserves (x and y in x*y=k).
        self.initial_token_a = initial_token_a
        self.initial_token_b = initial_token_b
        self.token_a = initial_token_a
        self.token_b = initial_token_b

        # Swap fee and microstructure controls.
        # NEW IN V4: arbitrage threshold + gas-like frictions + per-trade cap.
        self.fee = fee
        self.max_trade_fraction = max_trade_fraction
        self.arbitrage_band_bps = arbitrage_band_bps
        self.arb_gas_cost_b = arb_gas_cost_b
        self.rebalance_gas_cost_b = rebalance_gas_cost_b

        # AMM price and external reference market price series.
        self.market_price = self.get_price()
        self.price_history = [self.get_price()]
        self.market_price_history = [self.market_price]

        # Diagnostics used later for LP accounting and evaluation.
        self.volume_history = []
        self.fee_history = []
        self.arb_volume_history = []
        self.dislocation_history = [0.0]

        # Feed config and stochastic process config.
        self.step_count = 0
        self.price_feed = price_feed
        self.price_feed_index = 0
        self.market_volatility = 0.015
        self.market_drift = 0.0

    def get_price(self):
        """Current price of token A in terms of token B"""
        if self.token_a == 0:
            return float("inf")
        return self.token_b / self.token_a

    def _next_market_price(self, sentiment_signal=0.0):
        """Reference market price process (feed-driven or stochastic)."""
        # NEW IN V4: explicit external market process (AMM no longer moves alone).
        # If real prices are provided, consume them in order.
        if self.price_feed:
            if self.price_feed_index < len(self.price_feed):
                next_price = self.price_feed[self.price_feed_index]
                self.price_feed_index += 1
                if next_price > 0:
                    self.market_price = float(next_price)
                    return self.market_price

        # Otherwise use a stochastic process with occasional regime shifts.
        if self.step_count > 0 and self.step_count % 120 == 0:
            self.market_volatility = float(np.random.choice([0.008, 0.015, 0.03]))
            self.market_drift = np.random.normal(0, 0.0015)

        # Sentiment tilts drift but does not fully determine it.
        shock = np.random.normal(
            self.market_drift + sentiment_signal * 0.003,
            self.market_volatility,
        )
        self.market_price = max(1e-6, self.market_price * np.exp(shock))
        return self.market_price

    def _simulate_order_flow(self, sentiment_signal=0.0):
        """
        Simulates informed/uninformed flow as a stochastic trade each step.
        Returns (volume_b, fees_b).
        """
        # Trade sizes are noisy so we get calm periods and bursty periods.
        base_notional_b = self.token_b * 0.0025
        notional_b = base_notional_b * np.random.lognormal(mean=0.0, sigma=0.65)
        buy_prob = np.clip(0.5 + 0.35 * np.tanh(sentiment_signal), 0.05, 0.95)

        # Positive sentiment skews toward buying token A (B -> A swaps).
        if np.random.rand() < buy_prob:
            _, volume_b, fee_b = self.swap_b_for_a(notional_b)
        else:
            amount_a = notional_b / max(self.market_price, 1e-9)
            _, volume_b, fee_b = self.swap_a_for_b(amount_a)
        return volume_b, fee_b

    def _run_arbitrage(self):
        """
        If AMM price deviates from market price beyond a threshold, arbitrageurs
        trade toward the market price. Returns (arb_volume_b, arb_fees_b).
        """
        # NEW IN V4: arbitrage module links AMM to external market.
        amm_price = self.get_price()
        market_price = self.market_price
        if not np.isfinite(amm_price) or market_price <= 0:
            return 0.0, 0.0

        # Relative mispricing between pool and external market.
        dislocation = (amm_price - market_price) / market_price
        if abs(dislocation) * 10000 < self.arbitrage_band_bps:
            return 0.0, 0.0

        # Partial convergence per step to avoid unrealistic instant peg.
        target_price = amm_price + 0.8 * (market_price - amm_price)
        pool_k = self.token_a * self.token_b

        if target_price > amm_price:
            # Need to buy token A from pool by swapping B in.
            delta_b_effective = max(0.0, np.sqrt(pool_k * target_price) - self.token_b)
            amount_b_in = delta_b_effective / max(1 - self.fee, 1e-9)
            if amount_b_in < self.arb_gas_cost_b * 8:
                return 0.0, 0.0
            _, volume_b, fee_b = self.swap_b_for_a(amount_b_in)
        else:
            # Need to sell token A into pool to push price down.
            delta_a_effective = max(
                0.0,
                np.sqrt(pool_k / max(target_price, 1e-9)) - self.token_a,
            )
            amount_a_in = delta_a_effective / max(1 - self.fee, 1e-9)
            if amount_a_in * max(self.market_price, 1e-9) < self.arb_gas_cost_b * 8:
                return 0.0, 0.0
            _, volume_b, fee_b = self.swap_a_for_b(amount_a_in)

        # Arbitrage also contributes volume and fees for LPs.
        return volume_b, fee_b

    def simulate_market_activity(self, sentiment_signal=0.0):
        """
        One simulation step:
        1) Update reference market price
        2) Apply organic order flow
        3) Apply arbitrage flow if AMM dislocates from market
        """
        self.step_count += 1
        # Step 1: move the external market.
        self._next_market_price(sentiment_signal)

        # Step 2: organic trading flow.
        flow_volume_b, flow_fees_b = self._simulate_order_flow(sentiment_signal)
        # Step 3: arbitrage flow that corrects large dislocations.
        arb_volume_b, arb_fees_b = self._run_arbitrage()

        # Aggregate per-step execution stats.
        step_volume_b = flow_volume_b + arb_volume_b
        step_fees_b = flow_fees_b + arb_fees_b
        amm_price = self.get_price()
        dislocation = (amm_price - self.market_price) / max(self.market_price, 1e-9)

        # Persist histories for downstream state/features/plots.
        self.price_history.append(amm_price)
        self.market_price_history.append(self.market_price)
        self.volume_history.append(step_volume_b)
        self.fee_history.append(step_fees_b)
        self.arb_volume_history.append(arb_volume_b)
        self.dislocation_history.append(dislocation)

        return {
            "market_price": self.market_price,
            "amm_price": amm_price,
            "price_dislocation": dislocation,
            "volume_b": step_volume_b,
            "fees_collected_b": step_fees_b,
            "arb_volume_b": arb_volume_b,
        }

    def swap_a_for_b(self, amount_a):
        """
        Swap token A for token B.
        Returns (tokens_out_b, traded_notional_b, fee_value_b).
        """
        if amount_a <= 0:
            return 0.0, 0.0, 0.0

        # Cap trade size to keep single-step price impact bounded.
        amount_a = min(amount_a, self.token_a * self.max_trade_fraction)
        amount_a_with_fee = amount_a * (1 - self.fee)
        # Constant-product swap output formula.
        amount_b_out = self.token_b * amount_a_with_fee / (
            self.token_a + amount_a_with_fee
        )
        fee_a = amount_a - amount_a_with_fee

        # Gross input enters reserve; output leaves reserve.
        self.token_a += amount_a
        self.token_b -= amount_b_out

        # Convert fee into token-B notionals for consistent accounting.
        traded_notional_b = amount_a * max(self.market_price, 1e-9)
        fee_value_b = fee_a * max(self.market_price, 1e-9)
        return amount_b_out, traded_notional_b, fee_value_b

    def swap_b_for_a(self, amount_b):
        """
        Swap token B for token A.
        Returns (tokens_out_a, traded_notional_b, fee_value_b).
        """
        if amount_b <= 0:
            return 0.0, 0.0, 0.0

        # Cap trade size to avoid pathological one-step reserve depletion.
        amount_b = min(amount_b, self.token_b * self.max_trade_fraction)
        amount_b_with_fee = amount_b * (1 - self.fee)
        # Constant-product swap output formula.
        amount_a_out = self.token_a * amount_b_with_fee / (
            self.token_b + amount_b_with_fee
        )
        fee_b = amount_b - amount_b_with_fee

        # Gross input enters reserve; output leaves reserve.
        self.token_b += amount_b
        self.token_a -= amount_a_out

        # Here fee is already in token B.
        traded_notional_b = amount_b
        fee_value_b = fee_b
        return amount_a_out, traded_notional_b, fee_value_b

    def reset(self):
        # Restore clean initial state for a new episode.
        self.token_a = self.initial_token_a
        self.token_b = self.initial_token_b
        self.market_price = self.get_price()
        self.price_history = [self.get_price()]
        self.market_price_history = [self.market_price]
        self.volume_history = []
        self.fee_history = []
        self.arb_volume_history = []
        self.dislocation_history = [0.0]
        self.step_count = 0
        self.price_feed_index = 0


print("Section 1 done")

# %% [markdown]
# ## Section 2: Liquidity Provider Simulation
# Reference: `LiquidityProvider`

# %%


class LiquidityProvider:
    """
    Simulates a liquidity provider in an AMM pool.
    Tracks deposited value, current value, impermanent loss, and fees earned.
    """

    def __init__(self, deposit_a, deposit_b, entry_price):
        # Initial LP position snapshot.
        self.deposit_a = deposit_a
        self.deposit_b = deposit_b
        self.entry_price = entry_price
        self.initial_value = deposit_a * entry_price + deposit_b

        # Running PnL components (tracked in token-B value terms).
        self.fees_earned = 0.0
        self.costs_paid = 0.0
        self.total_deposited_value = self.initial_value

    def calculate_impermanent_loss(self, current_price):
        """Calculate impermanent loss based on price ratio"""
        if self.entry_price == 0:
            return 0
        price_ratio = current_price / self.entry_price
        if price_ratio <= 0:
            return 0
        # Standard constant-product IL formula relative to HODL baseline.
        il = 2 * np.sqrt(price_ratio) / (1 + price_ratio) - 1
        return il

    def calculate_position_value(
        self,
        current_price,
        active_exposure=1.0,
        trading_costs=0.0,
    ):
        """
        Current value with optional concentrated-liquidity exposure scaling
        and explicit transaction costs.
        """
        # NEW IN V4: active_exposure + trading_costs make valuation execution-aware.
        # In concentrated liquidity, only part of capital may be active/in-range.
        active_exposure = float(np.clip(active_exposure, 0.0, 1.5))
        il = self.calculate_impermanent_loss(current_price) * active_exposure
        # HODL baseline used as anchor before IL/fees/cost adjustments.
        hodl_value = self.deposit_a * current_price + self.deposit_b
        # Net LP value = IL-adjusted inventory + fees - execution/rebalance costs.
        lp_value = hodl_value * (1 + il) + self.fees_earned - trading_costs
        return lp_value

    def calculate_current_value(self, current_price, pool_share=1.0):
        """Backward-compatible wrapper used by older sections."""
        return self.calculate_position_value(current_price)

    def add_fees(self, fee_amount):
        # Called every step with LP share of pool fees.
        self.fees_earned += fee_amount

    def add_cost(self, cost_amount):
        # Optional explicit cost bucket for extensions.
        self.costs_paid += cost_amount


print("Section 2 done")

# %% [markdown]
# ## Section 3: Sentiment Analysis (Simulated)
# Reference: `SentimentAnalyzer`

# %%


class SentimentAnalyzer:
    """
    Simulates NLP-based sentiment analysis from market data.
    In a real implementation, this would use BERT/FinBERT on
    social media, news, and on-chain data.
    """

    def __init__(self, sentiment_feed=None):
        # Stores emitted signal for later diagnostics.
        self.sentiment_history = []
        # EMA-smoothed sentiment state.
        self.base_sentiment = 0.0
        # Optional deterministic feed for reproducible experiments.
        self.sentiment_feed = sentiment_feed
        self.sentiment_feed_index = 0

    def get_sentiment(self, price_history, window=10):
        """
        Generate a sentiment signal based on recent price action.
        Returns value in [-1, 1].
        """
        if self.sentiment_feed:
            # Feed mode: replay predefined sentiment path.
            if self.sentiment_feed_index < len(self.sentiment_feed):
                sentiment = self.sentiment_feed[self.sentiment_feed_index]
                self.sentiment_feed_index += 1
            else:
                sentiment = 0.0
        elif len(price_history) < 2:
            # Warm-up period: fallback noise if we have no momentum context.
            sentiment = np.random.normal(0, 0.1)
        else:
            recent = price_history[-min(window, len(price_history)) :]
            # Local return window used as proxy for narrative/mood drift.
            returns = [
                (recent[i] - recent[i - 1]) / recent[i - 1]
                for i in range(1, len(recent))
                if recent[i - 1] != 0
            ]

            if returns:
                momentum = np.mean(returns)
                volatility = np.std(returns) if len(returns) > 1 else 0.01
                # Sentiment based on momentum and noise (simulating NLP)
                nlp_noise = np.random.normal(0, 0.2)
                sentiment = (
                    np.tanh(momentum / max(volatility, 0.001) * 5) + nlp_noise
                )
            else:
                sentiment = np.random.normal(0, 0.1)

        # Smooth raw sentiment to avoid unrealistically jumpy policy inputs.
        self.base_sentiment = 0.7 * self.base_sentiment + 0.3 * sentiment
        final_sentiment = np.clip(self.base_sentiment, -1, 1)
        self.sentiment_history.append(final_sentiment)
        return final_sentiment


print("Section 3 done")

# %% [markdown]
# ## Section 4: DQL Agent for Liquidity Optimization
# Reference: `DQLAgent`

# %%


class DQLAgent:
    """
    Deep Q-Learning Agent for dynamic liquidity management.
    Uses a simple neural network approximation (tabular + function approx).
    """

    def __init__(self, state_size=8, action_size=5):
        # State/action dimensions are configurable for quick experimentation.
        self.state_size = state_size
        self.action_size = action_size

        # NEW IN V4: recenter actions approximate v3-style range management.
        # Actions: [hold, increase_liq, decrease_liq, recenter_up, recenter_down]
        self.action_names = [
            "Hold Position",
            "Increase Liquidity",
            "Decrease Liquidity",
            "Recenter Range Up",
            "Recenter Range Down",
        ]

        # Q-learning hyperparameters.
        self.gamma = 0.95  # discount factor
        self.epsilon = 1.0  # exploration rate
        self.epsilon_min = 0.05
        self.epsilon_decay = 0.995
        self.learning_rate = 0.001
        self.batch_size = 32

        # Replay memory stores transitions for off-policy updates.
        self.memory = deque(maxlen=2000)

        # Linear function approximator: Q(s,a) = s @ W + b.
        # This keeps notebook runtime light vs. a deep model.
        self.weights = np.random.randn(state_size, action_size) * 0.01
        self.bias = np.zeros(action_size)

        # Tracking values used for final diagnostics/plots.
        self.total_reward = 0
        self.reward_history = []

    def get_state(
        self,
        amm_price_history,
        market_price_history,
        sentiment,
        liquidity_ratio,
        il_current,
        fees_earned,
        volatility,
    ):
        """
        Construct state vector from market features.
        """
        # NEW IN V4: state now includes external market return + dislocation.
        # Short-horizon AMM price return.
        if len(amm_price_history) < 2:
            amm_return = 0.0
        else:
            amm_return = (
                (amm_price_history[-1] - amm_price_history[-2]) / amm_price_history[-2]
                if amm_price_history[-2] != 0
                else 0
            )

        # Short-horizon external market return.
        if len(market_price_history) < 2:
            market_return = 0.0
        else:
            market_return = (
                (market_price_history[-1] - market_price_history[-2])
                / market_price_history[-2]
                if market_price_history[-2] != 0
                else 0
            )

        current_market = market_price_history[-1] if market_price_history else 1.0
        current_amm = amm_price_history[-1] if amm_price_history else current_market
        # Positive value means AMM trades above external market.
        dislocation = (
            (current_amm - current_market) / max(current_market, 1e-9)
            if current_market > 0
            else 0.0
        )

        state = np.array(
            [
                amm_return,  # AMM return
                market_return,  # External market return
                sentiment,  # Market sentiment
                liquidity_ratio,  # Current liquidity allocation ratio
                il_current,  # Current impermanent loss
                fees_earned,  # Normalized fees earned
                volatility,  # Recent volatility
                dislocation,  # AMM-market gap
            ],
            dtype=float,
        )
        return state

    def predict_q_values(self, state):
        """Simple linear Q-value prediction"""
        return state @ self.weights + self.bias

    def act(self, state):
        """Epsilon-greedy action selection"""
        # Explore with probability epsilon, exploit otherwise.
        if np.random.rand() <= self.epsilon:
            return random.randrange(self.action_size)
        q_values = self.predict_q_values(state)
        return np.argmax(q_values)

    def remember(self, state, action, reward, next_state, done):
        """Store experience in replay buffer"""
        self.memory.append((state, action, reward, next_state, done))

    def replay(self):
        """Experience replay training"""
        # Wait until we have enough samples for a batch update.
        if len(self.memory) < self.batch_size:
            return

        minibatch = random.sample(self.memory, self.batch_size)

        # Standard one-step TD target.
        for state, action, reward, next_state, done in minibatch:
            target = reward
            if not done:
                target += self.gamma * np.max(self.predict_q_values(next_state))

            current_q = self.predict_q_values(state)
            error = target - current_q[action]

            # Update weights (gradient descent)
            self.weights[:, action] += self.learning_rate * error * state
            self.bias[action] += self.learning_rate * error

        # Decay exploration rate
        if self.epsilon > self.epsilon_min:
            self.epsilon *= self.epsilon_decay

    def calculate_reward(
        self,
        value_change,
        fees_gained,
        il_change,
        dislocation,
        rebalance_cost_ratio,
    ):
        """
        Cost-aware reward:
        - Reward net value growth and fee yield
        - Penalize rising IL, dislocation risk, and rebalance costs
        """
        # NEW IN V4: reward explicitly prices execution frictions and AMM-market gap.
        reward = (
            # Profitability objective.
            4.0 * value_change
            # Encourage fee capture when active and in-range.
            + 2.5 * fees_gained
            # Discourage increasing IL.
            - 4.0 * il_change
            # Penalize AMM-market mismatch risk.
            - 1.5 * abs(dislocation)
            # Penalize churn/recentering costs.
            - 2.0 * rebalance_cost_ratio
        )
        self.total_reward += reward
        self.reward_history.append(reward)
        return reward


print("Section 4 done")

# %% [markdown]
# ## Section 5: Simulation Engine
# Reference: `AIAMMSimulation`

# %%


def fetch_coingecko_ohlc(coin_id="ethereum", vs_currency="usd", days=1):
    # Optional real data source used when `use_real_data = True` in main().
    url = f"https://api.coingecko.com/api/v3/coins/{coin_id}/ohlc"
    params = {"vs_currency": vs_currency, "days": days}
    response = requests.get(url, params=params, timeout=30)
    response.raise_for_status()
    return response.json()


class AIAMMSimulation:
    """
    Main simulation engine that runs the AIAMM experiment.
    Compares static LP vs AI-driven LP management.
    """

    def __init__(
        self,
        num_steps=50,
        num_episodes=2,
        price_feed=None,
        sentiment_feed=None,
        seed=None,
    ):
        # Episode shape.
        self.num_steps = num_steps
        self.num_episodes = num_episodes
        # Optional deterministic feeds for external market and sentiment.
        self.price_feed = price_feed
        self.sentiment_feed = sentiment_feed
        # Seed both numpy and Python RNG for reproducibility.
        if seed is not None:
            np.random.seed(seed)
            random.seed(seed)
        # Placeholder for aggregate metrics.
        self.results = {
            "static": {"values": [], "il": [], "fees": [], "gap": []},
            "ai": {"values": [], "il": [], "fees": [], "gap": []},
        }

    @staticmethod
    def _pool_share(env, lp):
        # Simplified share approximation from initial reserve contribution.
        return min(
            lp.deposit_a / max(env.initial_token_a, 1e-9),
            lp.deposit_b / max(env.initial_token_b, 1e-9),
        )

    @staticmethod
    def _recent_volatility(series, window=12):
        # Return volatility over a recent rolling window.
        if len(series) < 2:
            return 0.01
        recent = series[-min(window, len(series)) :]
        returns = []
        for i in range(1, len(recent)):
            prev = recent[i - 1]
            if prev > 0:
                returns.append((recent[i] - prev) / prev)
        return float(np.std(returns)) if returns else 0.01

    @staticmethod
    def _active_fraction(market_price, range_center, range_half_width, liquidity_ratio):
        # Fraction of LP capital currently earning "full" fees in-range.
        # NEW IN V4: concentrated-liquidity style in-range/out-of-range behavior.
        if market_price <= 0 or range_center <= 0:
            return 0.0

        lower = range_center * (1 - range_half_width)
        upper = range_center * (1 + range_half_width)
        if lower <= market_price <= upper:
            return float(np.clip(liquidity_ratio, 0.0, 1.5))

        # Out-of-range liquidity degrades smoothly instead of dropping to zero.
        distance = abs(np.log(market_price / range_center))
        decay = np.exp(-8.0 * distance)
        return float(np.clip(liquidity_ratio * (0.1 + 0.4 * decay), 0.0, 1.5))

    def run_static_strategy(self):
        """Run a static (traditional) liquidity provision strategy."""
        # Baseline: LP does not change liquidity/range controls.
        env = AMMEnvironment(price_feed=self.price_feed)
        sentiment_analyzer = SentimentAnalyzer(sentiment_feed=self.sentiment_feed)

        lp = LiquidityProvider(
            deposit_a=1000,
            deposit_b=1000,
            entry_price=env.get_price(),
        )
        pool_share = self._pool_share(env, lp)

        values = []
        il_history = []
        dislocation_history = []

        for _ in range(self.num_steps):
            # Sentiment is generated from AMM price history proxy.
            sentiment = sentiment_analyzer.get_sentiment(env.price_history)
            # AMM + market co-evolution for one step.
            step_info = env.simulate_market_activity(sentiment)

            # Static LP receives its share of pool fees.
            lp.add_fees(step_info["fees_collected_b"] * pool_share)

            current_price = env.get_price()
            current_value = lp.calculate_position_value(current_price, active_exposure=1.0)
            il = lp.calculate_impermanent_loss(current_price)

            values.append(current_value)
            il_history.append(il)
            dislocation_history.append(step_info["price_dislocation"])

        return (
            values,
            il_history,
            lp.fees_earned,
            env.price_history,
            env.market_price_history,
            dislocation_history,
        )

    def run_ai_strategy(self):
        """Run AI-driven (DQL) liquidity management strategy."""
        # NEW IN V4: strategy controls both deployed liquidity and range centering.
        # Controlled environment with the same dynamics as baseline.
        env = AMMEnvironment(price_feed=self.price_feed)
        sentiment_analyzer = SentimentAnalyzer(sentiment_feed=self.sentiment_feed)
        agent = DQLAgent()

        lp = LiquidityProvider(
            deposit_a=1000,
            deposit_b=1000,
            entry_price=env.get_price(),
        )
        pool_share = self._pool_share(env, lp)

        # AI control knobs (simple proxy for concentrated-liquidity behavior).
        liquidity_ratio = 1.0
        range_center = env.market_price
        range_half_width = 0.12
        cumulative_rebalance_cost = 0.0

        values = []
        il_history = []
        active_history = []
        actions_taken = []
        prev_il = 0.0
        prev_value = lp.initial_value

        for step in range(self.num_steps):
            sentiment = sentiment_analyzer.get_sentiment(env.price_history)
            step_info = env.simulate_market_activity(sentiment)

            current_price = env.get_price()
            market_price = step_info["market_price"]
            il_raw = lp.calculate_impermanent_loss(current_price)
            vol = self._recent_volatility(env.market_price_history, window=12)

            # Estimate how much capital is active before action.
            active_fraction = self._active_fraction(
                market_price,
                range_center,
                range_half_width,
                liquidity_ratio,
            )
            il = il_raw * active_fraction

            # Build observation vector for RL policy.
            state = agent.get_state(
                env.price_history,
                env.market_price_history,
                sentiment,
                liquidity_ratio,
                il,
                lp.fees_earned / max(lp.initial_value, 1),
                vol,
            )

            action = agent.act(state)
            actions_taken.append(action)

            # Apply action effects + rebalance cost.
            action_cost = 0.0
            if action == 1:
                liquidity_ratio = min(liquidity_ratio + 0.1, 1.5)
                action_cost += env.rebalance_gas_cost_b
            elif action == 2:
                liquidity_ratio = max(liquidity_ratio - 0.1, 0.2)
                action_cost += env.rebalance_gas_cost_b
            elif action == 3:
                range_center *= 1.02
                action_cost += env.rebalance_gas_cost_b
            elif action == 4:
                range_center *= 0.98
                action_cost += env.rebalance_gas_cost_b

            # Passive drift: center slowly follows market even without recenter action.
            range_center = 0.98 * range_center + 0.02 * market_price
            cumulative_rebalance_cost += action_cost

            # Recompute active share after action.
            active_fraction = self._active_fraction(
                market_price,
                range_center,
                range_half_width,
                liquidity_ratio,
            )
            il = il_raw * active_fraction

            # Fee capture depends on pool fee generation and active LP coverage.
            step_fee = step_info["fees_collected_b"] * pool_share * active_fraction
            lp.add_fees(step_fee)

            # Net LP value after IL, fees, and cumulative rebalance costs.
            current_value = lp.calculate_position_value(
                current_price,
                active_exposure=active_fraction,
                trading_costs=cumulative_rebalance_cost,
            )

            # Transition reward terms.
            il_change = il - prev_il
            value_change = (current_value - prev_value) / max(prev_value, 1)
            fee_yield = step_fee / max(lp.initial_value, 1)
            reward = agent.calculate_reward(
                value_change=value_change,
                fees_gained=fee_yield,
                il_change=il_change,
                dislocation=step_info["price_dislocation"],
                rebalance_cost_ratio=action_cost / max(lp.initial_value, 1),
            )

            # Compute next state and train one replay step.
            next_state = agent.get_state(
                env.price_history,
                env.market_price_history,
                sentiment,
                liquidity_ratio,
                il,
                lp.fees_earned / max(lp.initial_value, 1),
                vol,
            )

            done = step == self.num_steps - 1
            agent.remember(state, action, reward, next_state, done)
            agent.replay()

            # Advance trackers used by value/IL series and plotting.
            prev_il = il
            prev_value = current_value
            values.append(current_value)
            il_history.append(il)
            active_history.append(active_fraction)

        return (
            values,
            il_history,
            lp.fees_earned,
            env.price_history,
            env.market_price_history,
            actions_taken,
            agent,
            active_history,
            env.dislocation_history,
        )

    def run_comparison(self):
        """Run full comparison simulation."""
        # Entry point used by main(): aggregate metrics across episodes.
        print("=" * 70)
        print("  AIAMM: AI Agents with AMMs for Smarter Liquidity Management")
        print("=" * 70)
        print()

        all_static_final = []
        all_ai_final = []
        all_static_il = []
        all_ai_il = []
        all_static_fees = []
        all_ai_fees = []
        all_static_gap = []
        all_ai_gap = []

        print(f"Running {self.num_episodes} simulation episodes...")
        print(f"Each episode: {self.num_steps} time steps")
        print("-" * 70)

        for episode in range(self.num_episodes):
            # Run baseline and AI policy under the same framework.
            s_values, s_il, s_fees, _, _, s_dislocation = self.run_static_strategy()
            all_static_final.append(s_values[-1])
            all_static_il.append(np.mean(s_il))
            all_static_fees.append(s_fees)
            all_static_gap.append(np.mean(np.abs(s_dislocation)) * 100)

            (
                a_values,
                a_il,
                a_fees,
                _,
                _,
                _,
                _,
                _,
                a_dislocation,
            ) = self.run_ai_strategy()
            all_ai_final.append(a_values[-1])
            all_ai_il.append(np.mean(a_il))
            all_ai_fees.append(a_fees)
            all_ai_gap.append(np.mean(np.abs(a_dislocation[1:])) * 100)

            # Progress marker for long runs.
            if (episode + 1) % 10 == 0:
                print(f"  Episode {episode + 1}/{self.num_episodes} completed")

        print("-" * 70)
        print()
        print("=" * 70)
        print("  RESULTS SUMMARY")
        print("=" * 70)
        print()

        print(
            f"{'Metric':<35} {'Static LP':>15} {'AI Agent':>15} {'Improvement':>15}"
        )
        print("-" * 80)

        # Portfolio value comparison.
        static_val = np.mean(all_static_final)
        ai_val = np.mean(all_ai_final)
        improvement_val = ((ai_val - static_val) / static_val) * 100
        print(
            f"{'Avg Final Portfolio Value ($)':<35} {static_val:>15.2f} "
            f"{ai_val:>15.2f} {improvement_val:>14.1f}%"
        )

        # IL comparison (percentage terms).
        static_il = np.mean(all_static_il) * 100
        ai_il = np.mean(all_ai_il) * 100
        il_reduction = (
            ((static_il - ai_il) / abs(static_il)) * 100 if static_il != 0 else 0
        )
        print(
            f"{'Avg Impermanent Loss (%)':<35} {static_il:>14.3f}% "
            f"{ai_il:>14.3f}% {il_reduction:>14.1f}%"
        )

        # Fee income comparison.
        static_fees = np.mean(all_static_fees)
        ai_fees = np.mean(all_ai_fees)
        fee_improvement = (
            ((ai_fees - static_fees) / static_fees) * 100 if static_fees != 0 else 0
        )
        print(
            f"{'Avg Fees Earned ($)':<35} {static_fees:>15.2f} "
            f"{ai_fees:>15.2f} {fee_improvement:>14.1f}%"
        )

        # Price efficiency comparison: AMM vs reference market.
        static_gap = np.mean(all_static_gap)
        ai_gap = np.mean(all_ai_gap)
        gap_reduction = (
            ((static_gap - ai_gap) / max(static_gap, 1e-9)) * 100 if static_gap > 0 else 0
        )
        print(
            f"{'Avg |AMM-Market Gap| (%)':<35} {static_gap:>14.3f}% "
            f"{ai_gap:>14.3f}% {gap_reduction:>14.1f}%"
        )

        print()
        print(
            f"{'Std Dev Portfolio Value ($)':<35} {np.std(all_static_final):>15.2f} "
            f"{np.std(all_ai_final):>15.2f}"
        )
        print(
            f"{'Std Dev Impermanent Loss (%)':<35} "
            f"{np.std(all_static_il)*100:>14.3f}% {np.std(all_ai_il)*100:>14.3f}%"
        )
        print()

        print("=" * 70)
        print("  DETAILED SINGLE EPISODE ANALYSIS")
        print("=" * 70)
        print()

        # Single-episode traces are used for visual diagnostics.
        (
            s_values,
            s_il,
            s_fees,
            s_price_hist,
            s_market_hist,
            _,
        ) = self.run_static_strategy()
        (
            a_values,
            a_il,
            a_fees,
            a_price_hist,
            _,
            actions,
            agent,
            active_history,
            a_dislocation,
        ) = self.run_ai_strategy()

        print("AI Agent Action Distribution:")
        print("-" * 40)
        for i in range(agent.action_size):
            count = actions.count(i)
            pct = count / len(actions) * 100
            bar = "#" * int(pct / 2)
            print(
                f"  {agent.action_names[i]:<25} {count:>5} ({pct:>5.1f}%) {bar}"
            )
        print()

        print("AI Agent Learning Metrics:")
        print("-" * 40)
        print(f"  Final Exploration Rate (epsilon): {agent.epsilon:.4f}")
        print(f"  Total Cumulative Reward:          {agent.total_reward:.2f}")
        print(f"  Memory Buffer Size:               {len(agent.memory)}")
        print()

        # Persist visualization for notebook and local inspection.
        self.generate_plots(
            s_values=s_values,
            a_values=a_values,
            s_il=s_il,
            a_il=a_il,
            s_price_hist=s_price_hist,
            a_price_hist=a_price_hist,
            s_market_hist=s_market_hist,
            actions=actions,
            agent=agent,
            active_history=active_history,
            a_dislocation=a_dislocation,
            s_fees=s_fees,
            a_fees=a_fees,
        )

        return {
            # Return a compact metrics dict for downstream use.
            "static_value": static_val,
            "ai_value": ai_val,
            "improvement": improvement_val,
            "static_il": static_il,
            "ai_il": ai_il,
            "il_reduction": il_reduction,
            "static_fees": static_fees,
            "ai_fees": ai_fees,
            "static_gap": static_gap,
            "ai_gap": ai_gap,
            "gap_reduction": gap_reduction,
        }

    def generate_plots(
        self,
        s_values,
        a_values,
        s_il,
        a_il,
        s_price_hist,
        a_price_hist,
        s_market_hist,
        actions,
        agent,
        active_history,
        a_dislocation,
        s_fees,
        a_fees,
    ):
        """Generate comparison plots."""
        # 2x3 layout: performance, risk, market structure, behavior.
        fig, axes = plt.subplots(2, 3, figsize=(18, 10))
        fig.suptitle(
            "AIAMM: AI-Driven vs Static Liquidity Management",
            fontsize=16,
            fontweight="bold",
        )

        # Plot 1: LP portfolio value trajectory.
        ax1 = axes[0, 0]
        ax1.plot(s_values, label="Static LP", alpha=0.8, color="blue")
        ax1.plot(a_values, label="AI Agent LP", alpha=0.8, color="green")
        ax1.set_title("Portfolio Value Over Time")
        ax1.set_xlabel("Time Step")
        ax1.set_ylabel("Value ($)")
        ax1.legend()
        ax1.grid(True, alpha=0.3)

        # Plot 2: IL path for static vs AI policy.
        ax2 = axes[0, 1]
        ax2.plot([x * 100 for x in s_il], label="Static LP IL", alpha=0.8, color="red")
        ax2.plot([x * 100 for x in a_il], label="AI Agent IL", alpha=0.8, color="orange")
        ax2.set_title("Impermanent Loss Over Time")
        ax2.set_xlabel("Time Step")
        ax2.set_ylabel("Impermanent Loss (%)")
        ax2.legend()
        ax2.grid(True, alpha=0.3)

        # Plot 3: AMM prices vs external reference.
        ax3 = axes[0, 2]
        ax3.plot(s_price_hist, color="tab:blue", alpha=0.7, label="Static AMM")
        ax3.plot(a_price_hist, color="tab:green", alpha=0.7, label="AI AMM")
        ax3.plot(
            s_market_hist,
            color="tab:gray",
            alpha=0.8,
            linestyle="--",
            label="Reference Market",
        )
        ax3.set_title("AMM vs Reference Price")
        ax3.set_xlabel("Time Step")
        ax3.set_ylabel("Price (Token B / Token A)")
        ax3.legend()
        ax3.grid(True, alpha=0.3)

        # Plot 4: Action frequency to inspect policy behavior.
        ax4 = axes[1, 0]
        action_counts = [actions.count(i) for i in range(agent.action_size)]
        colors = ["gray", "green", "red", "blue", "orange"]
        ax4.bar(range(agent.action_size), action_counts, color=colors, alpha=0.7)
        ax4.set_title("AI Agent Action Distribution")
        ax4.set_xlabel("Action")
        ax4.set_ylabel("Count")
        ax4.set_xticks(range(agent.action_size))
        ax4.set_xticklabels(
            ["Hold", "Inc Liq", "Dec Liq", "Up", "Down"],
            rotation=45,
            ha="right",
        )
        ax4.grid(True, alpha=0.3)

        # Plot 5: Running sum of rewards (learning signal).
        ax5 = axes[1, 1]
        cum_rewards = np.cumsum(agent.reward_history)
        ax5.plot(cum_rewards, color="green", alpha=0.8)
        ax5.set_title("AI Agent Cumulative Reward")
        ax5.set_xlabel("Time Step")
        ax5.set_ylabel("Cumulative Reward")
        ax5.grid(True, alpha=0.3)

        # Plot 6: Active liquidity and dislocation risk over time.
        ax6 = axes[1, 2]
        gap_series = np.abs(np.array(a_dislocation[1 : len(active_history) + 1])) * 100
        ax6.plot(
            np.array(active_history) * 100,
            label="Active Liquidity (%)",
            color="teal",
            alpha=0.8,
        )
        ax6.plot(
            gap_series,
            label="|AMM-Market Gap| (%)",
            color="tab:brown",
            alpha=0.8,
        )
        ax6.set_title("Execution Frictions and Range Activity")
        ax6.set_xlabel("Time Step")
        ax6.set_ylabel("Percent")
        ax6.legend()
        ax6.grid(True, alpha=0.3)

        plt.tight_layout()
        plt.savefig("aiamm_results.png", dpi=150, bbox_inches="tight")
        print("Visualization saved to 'aiamm_results.png'")
        print(f"Static fees: {s_fees:.2f} | AI fees: {a_fees:.2f}")
        print()


print("Section 5 done")

# %% [markdown]
# ## Section 6: Main Execution
# Reference: `main()` - run this cell to execute the simulation end-to-end.

# %%


def main():
    # Console banner for long-script runs.
    print()
    print("â•”â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•—")
    print("â•‘                                                                      â•‘")
    print("â•‘   AIAMM: Integrating AI Agents with Automated Market Makers          â•‘")
    print("â•‘          for Smarter Liquidity Management                            â•‘")
    print("â•‘                                                                      â•‘")
    print("â•‘   Author: Franco Farrugia                                            â•‘")
    print("â•‘   Research Area: Future Digital Technologies                         â•‘")
    print("â•‘                                                                      â•‘")
    print("â•‘   Algorithms: DQL | PPO | SAC (DQL demonstrated here)                â•‘")
    print("â•‘   Components: AMM Simulation | RL Agent | Sentiment Analysis         â•‘")
    print("â•‘                                                                      â•‘")
    print("â•šâ•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•")
    print()

    # Toggle between synthetic market path and CoinGecko replay.
    use_real_data = False
    price_feed = None
    if use_real_data:
        try:
            # OHLC format: [timestamp, open, high, low, close].
            ohlc_data = fetch_coingecko_ohlc("ethereum", "usd", days=1)
            # We use close prices as the reference market stream.
            price_feed = [row[4] for row in ohlc_data]
            print(f"Loaded {len(price_feed)} real price points.")
        except Exception as exc:
            # Fallback keeps notebook runnable if network/API is unavailable.
            print(f"Real data fetch failed, using simulation data instead: {exc}")
            price_feed = None

    # Large defaults provide smoother statistics but take longer to run.
    simulation = AIAMMSimulation(
        num_steps=5000,
        num_episodes=200,
        price_feed=price_feed,
        seed=42,
    )
    # Execute the full static-vs-AI comparison pipeline.
    results = simulation.run_comparison()
    print("Simulation complete.")
    print(results)


if __name__ == "__main__":
    main()


RUNNING FILE notebook
Section 0 done
Section 1 done
Section 2 done
Section 3 done
Section 4 done
Section 5 done

â•”â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•—
â•‘                                                                      â•‘
â•‘   AIAMM: Integrating AI Agents with Automated Market Makers          â•‘
â•‘          for Smarter Liquidity Management                            â•‘
â•‘                                                                      â•‘
â•‘   Author: Franco Farrugia                                            â•‘
â•‘   Research Area: Future Digital Technologies                         â•‘
â•‘                                                                      â•‘
â•‘   Algorithms: DQL | PPO | SAC (DQL demonstrated here)                â•‘
â•‘   Components: AMM Simulation | RL Agent | Sentimen